# Module 1 - Few-Shot Prompting

---

## What you will be able to craft from this module:

- A real understanding of what few-shot prompting is and why it works differently from zero-shot
- A mixed dataset built from real-world phishing data and AI-generated examples
- A fine-tuned phishing classifier trained on that dataset
- A side-by-side comparison of zero-shot vs few-shot vs your fine-tuned model
- A real production scenario grounded in an actual threat that companies face today

---

## Before you start

Make sure you have the following requirements ready:
- Completed Module 0 — some concepts here build directly on what we did there
- Python 3.10+
- A free Groq API key from [console.groq.com](https://console.groq.com)
- A `.env` file in the root of the repo with `GROQ_API_KEY=gsk_...`
- Dependencies installed: `pip install -r requirements.txt` from the root file

---

# The Scenario

You just started as the first dedicated internal security engineer at **Owlsight** — a 95-person cybersecurity SaaS company that sells threat monitoring and anomaly detection to mid-market enterprises.

The irony isn't lost on you. You sell security to others. Now you have to build it for yourself.

This is your first Monday. Before you even finish your coffee, three employees have forwarded you suspicious emails. One of them already clicked a link.

Your job today: build something that catches these before they reach the inbox.

---

## Why Owlsight is a high-value target

Owlsight isn't just any company. Their product ingests security telemetry from dozens of enterprise clients — SOC reports, vulnerability data, incident logs. Compromising Owlsight doesn't just hurt Owlsight. It gives an attacker a backdoor into every client they serve.

That makes them exactly the kind of target that sophisticated threat actors go after with precision. Not spray-and-pray phishing. Targeted, researched, personalized attacks.

And that's exactly what you're seeing in the three emails sitting in your inbox right now.

---

# The Threat — AI-Generated Phishing in 2025

## Why traditional filters are failing

For years, phishing detection relied on signals that are now completely dead:

- **Bad grammar and spelling** — LLMs write better English than most humans
- **Generic lures** — "Dear valued customer" is gone, replaced by emails that reference your actual job title, your manager's name, your recent projects
- **Suspicious formatting** — AI-generated emails match the exact tone and structure of legitimate company communications
- **Known malicious domains** — attackers now use compromised legitimate domains or generate convincing lookalikes

The old playbook doesn't work anymore. And the volume is only going up.

## What you are actually dealing with at Owlsight

The three emails forwarded to you this morning represent three different attack patterns:

**Email 1 — CEO impersonation (BEC)**
> *From: daniel.reyes@owlsight-corp.com*
> *"Hey, I need you to review this contract before our board call at 3pm. It's sensitive so please don't loop in anyone else yet. Link below."*

That domain is not owlsight.com. It's owlsight-**corp**.com. Registered yesterday.

**Email 2 — IT helpdesk impersonation**
> *From: security-ops@owlsight.com*
> *"We detected an unusual login to your account from São Paulo, Brazil. If this wasn't you, verify your identity immediately to prevent account suspension."*

Perfectly written. Correct branding. The link goes to a credential harvesting page.

**Email 3 — Vendor impersonation**
> *From: no-reply@docusign-secure.net*
> *"Your NDA with Meridian Capital Partners requires your signature by EOD. Click to review."*

Owlsight is actually in conversations with Meridian Capital. The attacker did their homework.

## Why few-shot is the right tool for this problem

Here's the challenge with phishing detection: **the threat evolves faster than you can retrain a model.**

A new phishing campaign emerges. You see 3 examples. You need your classifier to catch the next 10,000 variations of that same campaign — today, not after a two-week retraining cycle.

That's exactly what few-shot prompting solves. You show the model 2-3 examples of what the new attack looks like, and it immediately generalizes. No retraining. No new dataset. Just a prompt update.

This is why security teams at companies like Google, Microsoft, and CrowdStrike use few-shot prompting in their threat detection pipelines — not as a permanent solution, but as a rapid response layer while the fine-tuned model catches up.

---

# Wait — We Already Used Few-Shot in Module 0

Before we go further, let's pause on something.

Remember in Module 0 when we built the dataset generation prompt? We included this:

```python
[
  {"text": "I finally finished my thesis after 3 years!", "label": "happy"},
  {"text": "The meeting was rescheduled to Thursday.", "label": "neutral"},
  {"text": "I missed the last train home.", "label": "sad"}
]
```

Those three examples we put in the prompt? That **was** few-shot prompting. We were showing the model the format, the style, and the label structure before asking it to generate more.

We used it without naming it. Now we're going to use it deliberately — as a defense mechanism.

The difference between Module 0 and Module 1 is intention. In Module 0 few-shot was a tool to get better output format. In Module 1, few-shot is the actual technique we're deploying to solve a real security problem.

## So what exactly is few-shot prompting?

Few-shot prompting means providing a small number of input-output examples directly in your prompt before asking the model to perform the task. Instead of just describing what you want, you show it.

```
# Zero-shot:
"Is this email phishing or legitimate?"
Email: "Your account has been compromised. Click here."

# Few-shot:
"Is this email phishing or legitimate? Here are some examples:"

Email: "Please review Q3 report attached" → legitimate
Email: "Your password expires in 24h. Reset now." → phishing  
Email: "Hey, can we move our 1:1 to Thursday?" → legitimate

Now classify:
Email: "Your account has been compromised. Click here."
```

The examples anchor the model's behavior. They define what "phishing" and "legitimate" mean in your specific context — not in the abstract, but in the language and patterns of your actual environment.

## The "shot" terminology again

- **Zero-shot** = no examples, just the task description (Module 0)
- **One-shot** = one example per class
- **Few-shot** = typically 2-5 examples per class
- **Many-shot** = tens to hundreds of examples in the prompt (expensive, but powerful)
- **Fine-tuning** = training the model weights directly on your data (what we'll do later in this module)

The sweet spot for most production use cases is few-shot — enough examples to meaningfully guide the model, not so many that you're burning tokens on every request.

---

# Part 1 — Setting Up

> 💡 **Location of this code block?** Same setup as Module 0 — `shared/utils/api_helpers.py`. If you've already run Module 0 in this environment, your key is already loaded.

In [ ]:
import os
import json
import time
import random
import warnings
from pathlib import Path
from collections import Counter

from dotenv import load_dotenv
load_dotenv("../../../.env")

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise EnvironmentError(
        "GROQ_API_KEY not found. "
        "Make sure your .env file exists at the repo root and contains GROQ_API_KEY=gsk_..."
    )

print(f"✅ Groq API key loaded: {api_key[:8]}...")

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.1-8b-instant"

print(f"✅ Client ready — using model: {MODEL}")

---

# Part 2 — Finding Real-World Data

## Why we mix real data with generated data

In Module 0 we generated 100% of our dataset using Groq. That worked fine for sentiment analysis — the patterns are stable and universal.

Phishing is different. Real phishing emails have specific characteristics that a language model generating examples might not fully capture:
- Actual malicious URLs and domain patterns
- Real lure templates used by known threat actors
- The subtle language patterns that human analysts have flagged over years

At the same time, public phishing datasets have a blind spot: **they don't contain AI-generated phishing**, because most of them were collected before LLMs became widely used by attackers. That's where Groq comes in — we generate the synthetic AI-phishing examples that the real datasets are missing.

So our strategy is:
- **Real data** → traditional phishing and legitimate emails (from PhishTank + public corpora)
- **Generated data** → AI-crafted phishing emails in the Owlsight context

## Why PhishTank?

[PhishTank](https://phishtank.org) is a community-driven database of verified phishing URLs and email samples. It's been running since 2006, is free to use, and is one of the most widely referenced sources in academic phishing research.

**What makes it good for our use case:**
- Verified by human analysts, not just automated systems
- Covers a wide range of attack types and lure categories
- Updated continuously with new submissions
- Widely used in production security tools (you've probably been protected by something trained on PhishTank data)

**What it doesn't cover:**
- AI-generated phishing (too recent)
- Internal corporate lures specific to your environment
- Spear-phishing with personalized context

That gap is exactly what we'll fill with our Groq-generated Owlsight examples.

## Evaluating a dataset before using it

Before you use any public dataset in a production system, ask these questions:

1. **Who collected it and how?** Human-verified is better than automated. Community-verified is better than single-source.
2. **How old is it?** Phishing tactics from 2015 look very different from 2024. Stale data trains stale models.
3. **What's the label quality?** Are labels binary (phishing/not) or more granular? Do they cover the classes you care about?
4. **What's the class distribution?** A dataset with 95% phishing and 5% legitimate will produce a biased model.
5. **Is it representative of your environment?** A dataset of consumer banking phishing won't generalize well to B2B SaaS impersonation.

PhishTank scores well on 1, 2, and 3. We'll address 4 and 5 by balancing the dataset and adding our Owlsight-specific generated examples.

---

## Getting the PhishTank Dataset

You have two options here. **Option A** is the hands-on path — you go to the source, understand what you're downloading, and drop it in manually. **Option B** does it programmatically. Both end up in the same place.

If you're in a hurry, skip to Option B. But if you want to actually understand what's in your training data — and you should — Option A is worth the five minutes.

---

### Option A — Download it yourself (recommended)

1. Go to [phishtank.org/developer_info.php](https://phishtank.org/developer_info.php)
2. Create a free account and request an API key
3. Download `verified_online.json` — this is the current list of verified phishing URLs
4. Place the file at: `modules/01_few_shot/data/phishtank_raw.json`
5. Come back and run the cell below

While you're on the site, spend a few minutes browsing the submissions. Look at what kinds of URLs and targets are most common. That context will make the rest of this module more meaningful.

---

### Option B — Download programmatically

In [ ]:
import urllib.request

# PhishTank provides a public feed — no API key needed for the basic feed
# This downloads the current verified phishing URL database
PHISHTANK_URL = "http://data.phishtank.com/data/online-valid.json"
OUTPUT_PATH = Path("../data/phishtank_raw.json")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

if OUTPUT_PATH.exists():
    print(f"✅ PhishTank data already exists at {OUTPUT_PATH}")
    print("   Delete the file and re-run this cell if you want a fresh download.")
else:
    print("Downloading PhishTank verified phishing database...")
    print("This may take a minute — the file is several MB.\n")

    # PhishTank requires a User-Agent header or it blocks the request
    req = urllib.request.Request(
        PHISHTANK_URL,
        headers={"User-Agent": "phishtank/llm-prompting-lab"}
    )

    with urllib.request.urlopen(req) as response:
        data = response.read()

    with open(OUTPUT_PATH, "wb") as f:
        f.write(data)

    print(f"✅ Downloaded to {OUTPUT_PATH}")
    print(f"   File size: {OUTPUT_PATH.stat().st_size / 1024 / 1024:.1f} MB")

## Exploring the PhishTank data

Before we use any dataset, we look at it. Always. This is one of those habits that separates people who build reliable systems from people who wonder why their model behaves strangely six months later.

> 💡 **Location of this code block?** This is exploratory analysis — it lives in the notebook only. The cleaned output feeds into `scripts/01_generate_dataset.py`.

In [ ]:
# Load and inspect the PhishTank data
with open("../data/phishtank_raw.json") as f:
    phishtank_data = json.load(f)

print(f"Total entries: {len(phishtank_data)}")
print(f"\nSample entry (first record):")

# Look at what fields are available
sample = phishtank_data[0]
for key, value in sample.items():
    print(f"  {key}: {str(value)[:80]}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert to DataFrame for easier exploration
df_phish = pd.DataFrame(phishtank_data)

print("Columns available:")
print(df_phish.columns.tolist())
print(f"\nShape: {df_phish.shape}")
print(f"\nVerification status breakdown:")
print(df_phish["verified"].value_counts())

# Look at the target distribution — what are attackers impersonating?
if "target" in df_phish.columns:
    print(f"\nTop 15 impersonation targets:")
    print(df_phish["target"].value_counts().head(15))

## What you're seeing in the data

The PhishTank dataset gives us URLs and metadata, but not the full email body text — which is what we actually need to train a text classifier.

This is a common situation when working with real security datasets: they're collected for one purpose (URL reputation) and you need them for another (text classification). We're going to extract what's useful — the target, the URL pattern, the submission context — and use that to ground our Groq-generated examples in reality.

What we'll use from PhishTank:
- **Target** — who the attacker was impersonating (PayPal, Microsoft, a bank)
- **URL** — the domain pattern (we won't use real malicious URLs, but the pattern informs our generated examples)
- **Verification** — only using verified entries

We'll use these as seeds to generate realistic phishing email bodies with Groq.

In [ ]:
# Extract the most common targets to use as seeds for generation
# Filter to verified entries only
verified = df_phish[df_phish["verified"] == "yes"]

# Get top targets — these will anchor our generated phishing emails
if "target" in verified.columns:
    top_targets = verified["target"].value_counts().head(10)
    print("Top impersonation targets in verified PhishTank data:")
    for target, count in top_targets.items():
        print(f"  {target}: {count} entries")

    # Save these targets for use in generation
    PHISH_TARGETS = top_targets.index.tolist()
else:
    # Fallback if target column not available
    PHISH_TARGETS = ["PayPal", "Microsoft", "Apple", "Google", "Amazon",
                     "Chase Bank", "Wells Fargo", "Netflix", "LinkedIn", "Dropbox"]
    print("Using default targets:", PHISH_TARGETS)

print(f"\n✅ {len(PHISH_TARGETS)} targets ready for dataset generation")

---

# Part 3 — Building the Dataset

## Our three classes

We're classifying emails into three buckets:

- **`legitimate`** — real internal Owlsight emails, vendor communications, routine business
- **`traditional_phishing`** — the kind of phishing that's been around for years: generic lures, obvious urgency, impersonating well-known brands
- **`ai_phishing`** — the new wave: personalized, well-written, context-aware attacks that reference real details about Owlsight and its people

The distinction between the last two is the core of this module. Traditional phishing is a solved problem — most filters catch it. AI phishing is the unsolved problem — and that's what we're building a defense for.

## The Owlsight cast

Every generated email will reference real Owlsight context to make the examples feel production-grade:

- **Daniel Reyes** — CEO
- **Priya Nair** — Head of People & Culture
- **IT & Security Ops** — internal helpdesk
- **Vendors** — Google Workspace, Stripe, AWS, GitHub, DocuSign

> 💡 **Location of this code block?** This is the foundation of `scripts/01_generate_dataset.py` — specifically the prompt design and generation loop.

In [ ]:
# Owlsight company context — this gets injected into every generation prompt
# The more specific this is, the more realistic and useful your generated examples will be

OWLSIGHT_CONTEXT = """
Company: Owlsight — a 95-person cybersecurity SaaS company
Product: Threat monitoring and anomaly detection for mid-market enterprises
Email domain: @owlsight.com
People:
  - Daniel Reyes (CEO, daniel.reyes@owlsight.com)
  - Priya Nair (Head of People & Culture, priya.nair@owlsight.com)
  - IT & Security Ops (it-ops@owlsight.com)
Tools used: Google Workspace, Slack, GitHub, AWS, Stripe, DocuSign
Current events: Recently closed Series A, onboarding 3 new enterprise clients,
                hiring aggressively across engineering and sales
"""

print("✅ Owlsight context defined")

In [ ]:
# System prompts for each of our three classes
# Notice how each one has different instructions — the generated examples
# need to feel genuinely different from each other, not just relabeled versions
# of the same text

PROMPTS = {
    "legitimate": {
        "system": """You are generating realistic legitimate internal business emails for a cybersecurity company called Owlsight.
These are real emails that employees would actually send and receive — mundane, professional, and completely benign.
No urgency, no suspicious links, no requests for credentials.
Respond ONLY with a valid JSON array — no explanation, no markdown, no preamble.""",
        "user": """Generate {n} legitimate internal Owlsight emails.

Company context:
{context}

Each item must have exactly two fields: "text" and "label".
Label must be: "legitimate"

Include variety: meeting requests, project updates, HR announcements,
vendor invoices, onboarding messages, team celebrations, technical discussions.
Make them feel like real emails a real employee would write.

Example format:
[
  {{"text": "Hey team, just a reminder that our Q3 all-hands is this Friday at 2pm. Agenda will be shared tomorrow. — Daniel", "label": "legitimate"}}
]

Generate exactly {n} examples. Return ONLY the JSON array:"""
    },

    "traditional_phishing": {
        "system": """You are generating realistic examples of traditional phishing emails for a cybersecurity training dataset.
These are the classic phishing patterns that have existed for years — generic urgency, obvious impersonation of well-known brands,
grammatical tells, and unsophisticated lures.
This data will be used to train a phishing detection model. Generate realistic examples only.
Respond ONLY with a valid JSON array — no explanation, no markdown, no preamble.""",
        "user": """Generate {n} traditional phishing email examples.

These should represent the classic phishing patterns: impersonating PayPal, banks, Microsoft,
Apple, Google, Netflix, and similar well-known brands. Use generic urgency tactics like
account suspension, password expiry, suspicious login, prize winning, package delivery.

Top targets from real phishing data to use as inspiration: {targets}

Each item must have exactly two fields: "text" and "label".
Label must be: "traditional_phishing"

Example format:
[
  {{"text": "URGENT: Your PayPal account has been limited. Please verify your information immediately to restore access. Click here: paypa1-secure.com/verify", "label": "traditional_phishing"}}
]

Generate exactly {n} examples. Return ONLY the JSON array:"""
    },

    "ai_phishing": {
        "system": """You are generating realistic examples of AI-generated spear-phishing emails for a cybersecurity training dataset.
These represent the new wave of sophisticated phishing: personalized, well-written, context-aware attacks that reference
specific company details, real people's names, current events, and legitimate business contexts.
They are designed to bypass traditional filters and fool even security-aware employees.
This data will be used to train a phishing detection model. Generate realistic examples only.
Respond ONLY with a valid JSON array — no explanation, no markdown, no preamble.""",
        "user": """Generate {n} AI-generated spear-phishing email examples targeting Owlsight employees.

Company context:
{context}

These emails should be sophisticated and hard to spot:
- Impersonate Daniel Reyes (CEO), Priya Nair (HR), IT Security Ops, or trusted vendors
- Reference real Owlsight context (Series A, new clients, tools they use)
- Use subtle urgency without obvious red flags
- Include plausible but fake domains (owlsight-corp.com, owlsight.security-portal.com)
- Target credential harvesting, wire transfers, malware delivery, or data exfiltration

Each item must have exactly two fields: "text" and "label".
Label must be: "ai_phishing"

Example format:
[
  {{"text": "Hi, this is Daniel. I'm in back-to-back calls with the Meridian team but need you to review the updated NDA before 5pm. I've shared it here: docs.owlsight-secure.net/nda-v3 — please don't forward, it's under NDA itself. Thanks", "label": "ai_phishing"}}
]

Generate exactly {n} examples. Return ONLY the JSON array:"""
    }
}

print("✅ Generation prompts defined for all 3 classes")

In [ ]:
# Before we generate the full dataset, let's test each prompt with 2 examples
# This is good practice — always validate your prompts before a large generation run
# It saves time and API calls if something is off

def test_prompt(label: str, n: int = 2) -> list:
    """Test a generation prompt with a small batch."""
    config = PROMPTS[label]
    user_msg = config["user"].format(
        n=n,
        context=OWLSIGHT_CONTEXT,
        targets=", ".join(PHISH_TARGETS[:5])
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": config["system"]},
            {"role": "user",   "content": user_msg}
        ],
        temperature=0.9,
        max_tokens=1024,
    )

    raw = response.choices[0].message.content.strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    return json.loads(raw.strip())


print("Testing each prompt class...\n")

for label in ["legitimate", "traditional_phishing", "ai_phishing"]:
    examples = test_prompt(label, n=2)
    print(f"--- {label.upper()} ---")
    for ex in examples:
        print(f"  {ex['text'][:120]}..." if len(ex['text']) > 120 else f"  {ex['text']}")
    print()

## Reading the test output

Look at the three examples above and ask yourself:

- Does the **legitimate** email sound like something a real colleague would send?
- Does the **traditional phishing** email have the generic, obvious patterns you'd expect?
- Does the **AI phishing** email feel genuinely harder to spot? Does it reference Owlsight context?

If any of them feel off — too generic, too obvious, or mislabeled — you'd tweak the prompt before running the full generation. This is prompt engineering in practice: iterate on a small sample before committing to the full run.

In [ ]:
VALID_LABELS = {"legitimate", "traditional_phishing", "ai_phishing"}
BATCH_SIZE = 20


def generate_class(label: str, n: int) -> list:
    """
    Generate n examples for a single class.
    Runs in batches to stay within Groq's free tier rate limits.
    """
    config = PROMPTS[label]
    all_examples = []
    batch_num = 0
    failures = 0

    print(f"Generating {n} {label} examples...")

    while len(all_examples) < n:
        remaining = n - len(all_examples)
        this_batch = min(BATCH_SIZE, remaining)
        batch_num += 1

        try:
            user_msg = config["user"].format(
                n=this_batch,
                context=OWLSIGHT_CONTEXT,
                targets=", ".join(PHISH_TARGETS[:8]),
                third=this_batch // 3
            )

            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": config["system"]},
                    {"role": "user",   "content": user_msg}
                ],
                temperature=0.9,
                max_tokens=2048,
            )

            raw = response.choices[0].message.content.strip()
            if "```" in raw:
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]

            batch = json.loads(raw.strip())

            # Validate each example
            valid = [
                ex for ex in batch
                if isinstance(ex, dict)
                and "text" in ex
                and "label" in ex
                and ex["label"] == label  # enforce correct label
                and len(ex["text"].strip()) > 10
            ]

            all_examples.extend(valid)
            print(f"  Batch {batch_num}: {len(valid)} valid examples (total: {len(all_examples)}/{n})")

            if len(all_examples) < n:
                time.sleep(2)  # respect rate limits

        except json.JSONDecodeError:
            failures += 1
            print(f"  Parse error on batch {batch_num}, retrying...")
            time.sleep(3)
            if failures > 5:
                print("  Too many failures, saving what we have.")
                break

        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                print("  Rate limit hit — waiting 60s...")
                time.sleep(60)
            else:
                raise

    return all_examples[:n]


print("✅ Generation function ready")

In [ ]:
# Generate the full dataset
# 200 examples total: ~67 per class for a balanced distribution
# This will take about 2-3 minutes with the rate limit sleeps

EXAMPLES_PER_CLASS = 67

legitimate_examples      = generate_class("legitimate",          EXAMPLES_PER_CLASS)
traditional_phish        = generate_class("traditional_phishing", EXAMPLES_PER_CLASS)
ai_phish                 = generate_class("ai_phishing",          EXAMPLES_PER_CLASS)

# Combine and shuffle
dataset = legitimate_examples + traditional_phish + ai_phish
random.shuffle(dataset)

print(f"\n✅ Dataset ready: {len(dataset)} total examples")
label_counts = Counter(ex["label"] for ex in dataset)
for label, count in label_counts.items():
    print(f"  {label}: {count}")

In [ ]:
# Save the full dataset and create splits
# Same pattern as Module 0 — 80% train, 10% val, 10% test

data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

def save_jsonl(examples: list, path: Path):
    with open(path, "w") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"  Saved {len(examples):4d} examples → {path}")


save_jsonl(dataset, data_dir / "phishing_raw.jsonl")

n = len(dataset)
train_end = int(n * 0.8)
val_end   = train_end + int(n * 0.1)

print("\nSaving splits:")
save_jsonl(dataset[:train_end],  data_dir / "phishing_train.jsonl")
save_jsonl(dataset[train_end:val_end], data_dir / "phishing_val.jsonl")
save_jsonl(dataset[val_end:],    data_dir / "phishing_test.jsonl")

print(f"\n✅ All data saved to {data_dir.resolve()}")

---

# Part 4 — The Core Few-Shot Demonstration

This is the heart of the module. Before we train anything, we're going to show you exactly what few-shot prompting does — and why it matters for this specific problem.

We'll run three experiments on the same set of test emails:
1. **Zero-shot** — just tell the model what to do, no examples
2. **Few-shot** — give it 2-3 examples per class before asking
3. **We'll add the fine-tuned model in Part 5**

The emails we'll test on are designed to stress the hardest case: **AI-generated phishing**. These are the ones that fool people.

In [ ]:
# A set of hand-crafted test emails representing the Owlsight scenario
# These are deliberately challenging — some obvious, some very subtle

TEST_EMAILS = [
    # --- Legitimate ---
    {"text": "Hey, quick heads up — the 3pm client call has been moved to 4pm. Same link. — Priya",
     "label": "legitimate"},
    {"text": "Team, we're enabling mandatory MFA for all GitHub repos starting Monday. IT-Ops will send setup instructions to your work email.",
     "label": "legitimate"},
    {"text": "Hi everyone, excited to announce we've officially signed our third enterprise client this quarter. Details in the all-hands Friday. — Daniel",
     "label": "legitimate"},

    # --- Traditional Phishing ---
    {"text": "YOUR ACCOUNT HAS BEEN SUSPENDED. Click here immediately to verify your identity and restore access: secure-paypa1.com/verify",
     "label": "traditional_phishing"},
    {"text": "Congratulations! You have been selected to receive a $500 Amazon gift card. Claim your reward now before it expires.",
     "label": "traditional_phishing"},
    {"text": "Dear Customer, your Microsoft account password will expire in 24 hours. Update it now to avoid losing access to your files.",
     "label": "traditional_phishing"},

    # --- AI Phishing (the hard ones) ---
    {"text": "Hey, it's Daniel. I'm in back-to-back calls with the Meridian team but need you to review the updated NDA before 5pm. Shared it here: docs.owlsight-secure.net/nda — please don't forward, it's confidential. Thanks",
     "label": "ai_phishing"},
    {"text": "Hi, this is Priya from People & Culture. As part of our Series A onboarding updates, we need all employees to re-verify their benefits enrollment by EOD. Use the HR portal: owlsight-hr.people-portal.com",
     "label": "ai_phishing"},
    {"text": "Security alert: we detected a login to your Owlsight GitHub account from an unrecognized device in Amsterdam. If this wasn't you, revoke access immediately: github-owlsight.security-verify.com/revoke",
     "label": "ai_phishing"},
]

print(f"✅ {len(TEST_EMAILS)} test emails ready")
print("\nClass distribution:")
for label, count in Counter(ex["label"] for ex in TEST_EMAILS).items():
    print(f"  {label}: {count}")

In [ ]:
# Experiment 1: Zero-shot classification
# No examples — just a description of the task
# This is exactly what we did in Module 0, just applied to a security problem

ZERO_SHOT_SYSTEM = """You are a phishing detection system for Owlsight, a cybersecurity company.
Classify emails into exactly one of three categories:
- legitimate: real internal business email
- traditional_phishing: generic phishing with obvious red flags
- ai_phishing: sophisticated, personalized phishing that is hard to spot

Respond with exactly one of these three labels and nothing else."""


def classify_zero_shot(email_text: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": ZERO_SHOT_SYSTEM},
            {"role": "user",   "content": f"Classify this email:\n\n{email_text}"}
        ],
        temperature=0,
        max_tokens=10,
    )
    pred = response.choices[0].message.content.strip().lower()
    # Normalize the response in case the model adds extra words
    for label in ["ai_phishing", "traditional_phishing", "legitimate"]:
        if label in pred:
            return label
    return "legitimate"  # fallback


print("Running zero-shot classification on all test emails...\n")
zero_shot_results = []

for ex in TEST_EMAILS:
    pred = classify_zero_shot(ex["text"])
    correct = "✅" if pred == ex["label"] else "❌"
    zero_shot_results.append(pred)
    print(f"{correct} [{ex['label']:20s}] → predicted: {pred}")
    time.sleep(0.5)

zero_shot_acc = sum(p == e["label"] for p, e in zip(zero_shot_results, TEST_EMAILS)) / len(TEST_EMAILS)
print(f"\nZero-shot accuracy: {zero_shot_acc*100:.0f}%")

## What you're seeing

Pay attention to which emails the zero-shot model gets wrong. The traditional phishing emails are probably fine — those patterns are well-represented in the model's pretraining data. 

The AI phishing emails are where it gets interesting. The model has never been shown what Owlsight-specific AI phishing looks like. It has no examples to anchor its classification. It's making its best guess from a general description.

Now let's give it a few examples and see what changes.

In [ ]:
# Experiment 2: Few-shot classification
# We provide 2 examples per class before asking the model to classify
# These examples come from our generated dataset — realistic Owlsight emails

# Pick 2 examples per class from our generated dataset to use as shots
# We use examples that are NOT in our test set
def get_few_shot_examples(dataset: list, n_per_class: int = 2) -> list:
    examples = []
    for label in ["legitimate", "traditional_phishing", "ai_phishing"]:
        class_examples = [ex for ex in dataset if ex["label"] == label]
        examples.extend(random.sample(class_examples, min(n_per_class, len(class_examples))))
    return examples


few_shot_examples = get_few_shot_examples(dataset, n_per_class=2)

# Build the few-shot portion of the prompt
# This is the key difference from zero-shot — we're showing, not just telling
few_shot_block = "\n".join([
    f'Email: "{ex["text"][:200]}"\nLabel: {ex["label"]}\n'
    for ex in few_shot_examples
])

FEW_SHOT_SYSTEM = f"""You are a phishing detection system for Owlsight, a cybersecurity company.
Classify emails into exactly one of three categories:
- legitimate: real internal business email
- traditional_phishing: generic phishing with obvious red flags
- ai_phishing: sophisticated, personalized phishing that is hard to spot

Here are examples of each class:

{few_shot_block}
Respond with exactly one of the three labels and nothing else."""

print("Few-shot prompt built with examples:")
for ex in few_shot_examples:
    print(f"  [{ex['label']:20s}] {ex['text'][:80]}...")

In [ ]:
def classify_few_shot(email_text: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": FEW_SHOT_SYSTEM},
            {"role": "user",   "content": f"Classify this email:\n\n{email_text}"}
        ],
        temperature=0,
        max_tokens=10,
    )
    pred = response.choices[0].message.content.strip().lower()
    for label in ["ai_phishing", "traditional_phishing", "legitimate"]:
        if label in pred:
            return label
    return "legitimate"


print("Running few-shot classification on all test emails...\n")
few_shot_results = []

for ex in TEST_EMAILS:
    pred = classify_few_shot(ex["text"])
    correct = "✅" if pred == ex["label"] else "❌"
    few_shot_results.append(pred)
    print(f"{correct} [{ex['label']:20s}] → predicted: {pred}")
    time.sleep(0.5)

few_shot_acc = sum(p == e["label"] for p, e in zip(few_shot_results, TEST_EMAILS)) / len(TEST_EMAILS)
print(f"\nFew-shot accuracy: {few_shot_acc*100:.0f}%")
print(f"Zero-shot accuracy was: {zero_shot_acc*100:.0f}%")
print(f"\nImprovement from just adding examples to the prompt: {(few_shot_acc - zero_shot_acc)*100:.0f} percentage points")

## What just happened

We didn't change the model. We didn't retrain anything. We didn't add new weights or update any parameters.

We added 6 examples to the prompt — 2 per class — and the model's behavior changed.

That's the power of few-shot prompting. The examples act as an anchor. They tell the model: in *this* context, with *these* specific patterns, here is what each class looks like. The model uses those examples to recalibrate its priors on the fly.

For a security team, this means you can respond to a new phishing campaign the same day you see it. You collect 2-3 examples, add them to your prompt, and your detection improves immediately — no retraining cycle, no waiting for a new model deployment.

That's the real-world value. Now let's see how it compares to a properly fine-tuned model.

---

# Part 5 — Fine-Tuning the Classifier

Same pattern as Module 0 — we take a pretrained DistilBERT and add a classification head for our three classes. The difference here is what we're classifying and what the stakes are.

In Module 0, getting a sentiment wrong costs nothing. Here, a false negative (calling a phishing email legitimate) means an employee clicks a malicious link. That changes how you think about the evaluation metrics.

> 💡 **Location of this code block?** This maps directly to `scripts/02_train.py`. Same architecture as Module 0, updated for 3 new classes.

> ⚠️ **Expected warning:** When loading DistilBERT you'll see a LOAD REPORT with UNEXPECTED and MISSING keys. This is normal — we covered this in Module 0. The MISSING keys are our new classification head being randomly initialized before training.

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    logging as transformers_logging,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

transformers_logging.set_verbosity_error()

LABEL2ID = {"legitimate": 0, "traditional_phishing": 1, "ai_phishing": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

In [ ]:
class PhishingDataset(Dataset):
    """
    Same structure as Module 0's SentimentDataset.
    The pattern is identical — load JSONL, tokenize, return tensors.
    Only the label mapping changes.
    """
    def __init__(self, path: str, tokenizer, max_length: int = 256):
        # max_length=256 instead of 128 — emails are longer than sentiment sentences
        self.examples = []
        with open(path) as f:
            for line in f:
                ex = json.loads(line.strip())
                if ex.get("label") in LABEL2ID:
                    self.examples.append(ex)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        encoding = self.tokenizer(
            ex["text"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(LABEL2ID[ex["label"]], dtype=torch.long),
        }


tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_ds = PhishingDataset("../data/phishing_train.jsonl", tokenizer)
val_ds   = PhishingDataset("../data/phishing_val.jsonl",   tokenizer)
test_ds  = PhishingDataset("../data/phishing_test.jsonl",  tokenizer)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

In [ ]:
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16)
test_loader  = DataLoader(test_ds,  batch_size=16)

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)

EPOCHS = 5
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")
print(f"Training for {EPOCHS} epochs ({total_steps} total steps)")

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    return total_loss / len(dataloader), acc, f1


print("evaluate() ready. Starting training loop...\n")

In [ ]:
checkpoints_dir = Path("../checkpoints")
checkpoints_dir.mkdir(exist_ok=True)

best_val_f1 = 0.0
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for step, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc, val_f1 = evaluate(model, val_loader)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch}/{EPOCHS}  |  train loss: {avg_train_loss:.4f}  |  val acc: {val_acc:.4f}  |  val F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained(checkpoints_dir / "best_model")
        tokenizer.save_pretrained(checkpoints_dir / "best_model")
        print(f"  💾 New best model saved (F1={val_f1:.4f})")

print(f"\n✅ Training complete. Best val F1: {best_val_f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt

figures_dir = Path("../figures")
figures_dir.mkdir(exist_ok=True)

epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], "o-", label="Train loss", color="#3498db")
axes[0].plot(epochs_range, history["val_loss"],   "o-", label="Val loss",   color="#e74c3c")
axes[0].set_title("Loss"), axes[0].set_xlabel("Epoch")
axes[0].legend(), axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["val_acc"], "o-", label="Val accuracy", color="#2ecc71")
axes[1].plot(epochs_range, history["val_f1"],  "o-", label="Val F1",       color="#9b59b6")
axes[1].set_title("Validation Metrics"), axes[1].set_xlabel("Epoch")
axes[1].legend(), axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

---

# Part 6 — The Full Comparison

Now we have all three approaches ready. Let's run them side by side on the same test emails and see what we're actually dealing with.

Remember: in a real Owlsight deployment, every false negative is an employee who gets a phishing email delivered to their inbox unchallenged. That framing matters when you read these numbers.

> 💡 **Location of this code block?** This is `scripts/03_evaluate.py` — the full comparison logic lives there.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Load best model
best_model = AutoModelForSequenceClassification.from_pretrained(
    checkpoints_dir / "best_model"
).to(device)

# Run fine-tuned model on the hand-crafted test emails
best_model.eval()
ft_results = []

for ex in TEST_EMAILS:
    encoding = tokenizer(
        ex["text"],
        max_length=256,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = best_model(**encoding)

    pred_id = torch.argmax(outputs.logits, dim=-1).item()
    ft_results.append(ID2LABEL[pred_id])

ft_acc = sum(p == e["label"] for p, e in zip(ft_results, TEST_EMAILS)) / len(TEST_EMAILS)

# Print the full side-by-side comparison
print(f"{'Email (truncated)':<45} {'True':>20} {'Zero-shot':>15} {'Few-shot':>15} {'Fine-tuned':>12}")
print("-" * 110)

for i, ex in enumerate(TEST_EMAILS):
    text_short = ex["text"][:43] + "..."
    zs = "✅" if zero_shot_results[i] == ex["label"] else "❌"
    fs = "✅" if few_shot_results[i]  == ex["label"] else "❌"
    ft = "✅" if ft_results[i]        == ex["label"] else "❌"
    print(f"{text_short:<45} {ex['label']:>20} {zs+' '+zero_shot_results[i]:>15} {fs+' '+few_shot_results[i]:>15} {ft+' '+ft_results[i]:>12}")

print("\n" + "=" * 60)
print("  FINAL COMPARISON")
print("=" * 60)
print(f"  {'Zero-shot Groq':<30} {zero_shot_acc*100:.0f}%")
print(f"  {'Few-shot Groq':<30} {few_shot_acc*100:.0f}%")
print(f"  {'Fine-tuned DistilBERT':<30} {ft_acc*100:.0f}%")
print("=" * 60)

## Reading the final comparison

Three approaches, same emails, very different results depending on what you're optimizing for.

**Zero-shot** is your baseline — what you get before doing any work. It knows what phishing is in general, but it has no idea what Owlsight-specific AI phishing looks like.

**Few-shot** is your rapid response layer — what you can deploy in minutes when a new campaign hits. You collected a few examples, added them to the prompt, and the model's behavior changed immediately. No retraining, no new deployment.

**Fine-tuned** is your production model — slower to update but faster at inference, cheaper at scale, and specialized to your exact environment.

In a real Owlsight deployment, you'd use all three together:
- Fine-tuned model as the primary filter, running on every email
- Few-shot prompt as the rapid response layer when a new campaign appears
- Zero-shot as the fallback for categories the fine-tuned model hasn't seen yet

That's not a theoretical architecture. That's how production email security systems at scale actually work.

---

# Key Takeaways of this module

## What we just built

1. **We grounded the problem in a real scenario** — Owlsight, a security company getting targeted by the exact kind of attack they protect others from
2. **We mixed real and generated data** — PhishTank for traditional phishing patterns, Groq for the AI-generated phishing that public datasets don't cover yet
3. **We demonstrated few-shot prompting as a rapid response mechanism** — not just a prompting technique, but a practical tool for security teams
4. **We built the full pipeline** — from raw data to fine-tuned classifier to production comparison

## The most important thing to take back home

Few-shot prompting is not a workaround for not having enough data. It's a deliberate strategy for situations where the threat evolves faster than you can retrain.

In cybersecurity, that's almost always the situation. Attackers adapt in days. Model retraining cycles take weeks. The gap between those two timelines is where few-shot lives — and now you know how to use it.

## Ready to keep testing?

- Add a 4th class: `business_email_compromise` — wire transfer fraud, vendor impersonation. How does the model handle it with 3 new examples?
- Try increasing the few-shot examples from 2 to 5 per class — does accuracy improve? At what point do you hit diminishing returns?
- Look at your model's false negatives on AI phishing specifically — what do those emails have in common? What would you add to the training data?
- Try the same few-shot prompt with a larger model on Groq (`llama-3.1-70b-versatile`) — how much does model size matter for this task?

---

## Next module

**[Module 2 — Chain-of-Thought Prompting →](../../02_chain_of_thought/notebooks/02_chain_of_thought.ipynb)**

We stay at Owlsight. A new challenge: not just classifying threats, but explaining them. You'll teach a model to reason step by step through an attack, producing analyst-ready threat summaries that a human SOC team can actually act on.